# Assessment card demo (fusion-v3, frozen system)

This notebook renders assessment cards for the frozen fusion-v3 system (freeze commit `8f351b6`).
It only reads precomputed outputs in `demo/card_data/`: no model is fitted or run, and no metric is computed.
Those outputs were written from the frozen artefacts by `src/card/build_card_data.py`, `build_kan_terms.py` and `build_gradcam.py` (see `demo/README.md`).

**Research prototype, not for clinical use.** Train and val subjects were in the frozen system's training data, so their outputs are in-sample. Test subjects are held out; their outputs are those of the single frozen test run.

In [ ]:
import sys, time
from pathlib import Path

# repo root = the first directory upwards that contains src/card (no absolute paths)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "card").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from card import CardData, render_card, select_showcase

t0 = time.time()
OUT = ROOT / "demo" / "output"
data = CardData()
print(f"{len(data.ptids)} subjects:", data.subjects.split.value_counts().to_dict())

## Showcase set

The rule is fixed in `src/card/showcase.py` and was committed before any card was generated. From the val split, it takes the first subject by ID in each category:
- correct CN, EMCI, LMCI and AD;
- a misclassified LMCI;
- a subject with the discordance flag;
- a subject with an unreliable brain-age scan;
- the subject with no scan.

A category with no val subject falls back to train. The unreliable-scan category does: no val scan fails the rule.

In [ ]:
showcase = select_showcase(data.table())
showcase

In [ ]:
paths = []
for _, r in showcase.iterrows():
    h, p = render_card(data, r.PTID, OUT, r.category + (" (fallback: train split)" if r.fallback else ""))
    paths.append((r.category, r.PTID, h, p))
for c, ptid, h, p in paths:
    print(f"{c:28s} {ptid}  {Path(h).relative_to(ROOT)}  {Path(p).relative_to(ROOT)}")

In [ ]:
from IPython.display import Image, display
display(Image(filename=paths[4][3], width=800))   # the misclassified LMCI card

## Any subject

Set `SUBJECT` to any train, val or test PTID and re-run the cell. The cell also offers a picker when ipywidgets is available.
It reads precomputed outputs only, so each card takes about a second.

In [ ]:
SUBJECT = data.subjects[data.subjects.split == "test"].PTID.iloc[0]   # first test subject by ID; any PTID works

def show(ptid):
    h, p = render_card(data, ptid, OUT)
    print(Path(h).relative_to(ROOT))
    display(Image(filename=p, width=800))

show(SUBJECT)

In [ ]:
# Optional picker (interactive Jupyter only): type or choose any of the 814 PTIDs.
try:
    import ipywidgets as w
    box = w.Combobox(placeholder="PTID", options=data.ptids, ensure_option=True, description="PTID")
    out = w.Output()
    def on_change(ch):
        if ch["new"] in data.index:
            out.clear_output()
            with out:
                show(ch["new"])
    box.observe(on_change, names="value")
    display(w.VBox([box, out]))
except ImportError:
    print("ipywidgets not installed: set SUBJECT above instead")

In [ ]:
print(f"done in {time.time() - t0:.1f} s")